In [67]:
# imports
import os
import requests
import re
import pandas as pd 
from tqdm import tqdm

from transformers import AutoTokenizer
import datasets
from datasets import concatenate_datasets, Dataset
import openai


## Download Papers

In [2]:
dataset = datasets.load_dataset(
    "LeMaterial/LeMat-Synth-Papers",
    "full"
    )

In [3]:
print(dataset.column_names)

{'arxiv': ['id', 'title', 'authors', 'abstract', 'doi', 'published_date', 'updated_date', 'categories', 'license', 'pdf_url', 'views_count', 'read_count', 'citation_count', 'keywords', 'text_paper', 'text_si', 'source', 'pdf_extractor', 'images', 'structured_synthesis'], 'omg24': ['id', 'title', 'authors', 'abstract', 'doi', 'published_date', 'updated_date', 'categories', 'license', 'pdf_url', 'views_count', 'read_count', 'citation_count', 'keywords', 'text_paper', 'text_si', 'source', 'pdf_extractor', 'images', 'structured_synthesis'], 'chemrxiv': ['id', 'title', 'authors', 'abstract', 'doi', 'published_date', 'updated_date', 'categories', 'license', 'pdf_url', 'views_count', 'read_count', 'citation_count', 'keywords', 'text_paper', 'text_si', 'source', 'pdf_extractor', 'images', 'structured_synthesis']}


In [4]:
concated_data = concatenate_datasets([dataset['chemrxiv'],dataset['omg24'], dataset['arxiv']])

In [5]:
len(concated_data)

80940

## Filter data

In [52]:
df = concated_data.to_pandas()
df = df.drop_duplicates(subset='doi', keep='first')
all_data = Dataset.from_pandas(df)
all_data = all_data.remove_columns(['__index_level_0__'])

In [53]:
len(all_data)

46459

In [55]:
# Define keywords (all lowercase)
keywords = ['catalysis', 'catalytic', 'catalyst', 'activation energy', 'TOF']

# Compile regular expression for fast search
pattern = re.compile(r'\b(' + '|'.join(re.escape(k) for k in keywords) + r')\b', flags=re.IGNORECASE)

def keyword_in_abstract(example):
    text = example.get('text_paper', '') or ''
    return bool(pattern.search(text))

catalysis_papers = all_data.filter(keyword_in_abstract)

Filter: 100%|██████████| 46459/46459 [01:26<00:00, 539.44 examples/s]


In [56]:
len(catalysis_papers)

7144

## LLM Filtering

In [58]:
prompt = """You are provided with a paper. We want to know if the paper uses a catalytic process to synthesize the material. Answer with only yes or no. 
Start Example: 
Paper: [paper_text]
Question: Does this paper use a catalytic process to synthesize the material?
Answer: [yes/no]
End Example.


Paper: {paper_text}
Question: Does this paper use a catalytic process to synthesize the material?
Answer: 
"""

In [59]:
# Assume you have an OpenAI-compatible endpoint, e.g.:
def ask_llm_is_catalytic(text, client, model):
    """
    Query an LLM to determine if a paper uses a catalytic process to synthesize the material.
    Returns True if catalytic, False if not, None if unclear.
    Uses the provided client to make the API call.
    """
    message = prompt.format(paper_text=text)
    try:
        # Use the provided client to make the LLM API request (OpenAI-compatible)
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "user", "content": message}
            ],
            temperature=0,
            max_tokens=100,
        )
        # Try to extract the answer in the same way as before
        answer = response.choices[0].message.content.strip().lower()
        if answer in ['yes', 'yes.']:
            return True
        elif answer in ['no', 'no.']:
            return False
        elif 'yes' in answer:
            return True
        elif 'no' in answer:
            return False
        else:
            return False
    except Exception as e:
        print(f"LLM call failed: {e}")
        return False

In [60]:
# create openai client
client = openai.OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")
model_name = "mistralai/Mistral-Small-24B-Instruct-2501"
max_model_len = 32768
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# Example usage (evaluate for first 5 papers)
# results = [ask_llm_is_catalytic(p['text_paper']) for p in catalysis_papers.select(range(5))]

from concurrent.futures import ThreadPoolExecutor, as_completed

def preprocess_text(example):
    # tokenize the text, check if it's more than max_model_len, if so, split it into chunks, take the first chunk and pass it to the LLM
    text = example['text_paper']
    tokens = tokenizer.encode(text)
    if len(tokens) > max_model_len:
        text = tokenizer.decode(tokens[:max_model_len-150])
    return text

def process_example(example):
    text = preprocess_text(example)
    return ask_llm_is_catalytic(text, client, model_name)

results = []
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {executor.submit(process_example, example): i for i, example in enumerate(catalysis_papers)}
    for f in tqdm(as_completed(futures), total=len(catalysis_papers), desc="LLM Catalysis Check"):
        results.append(f.result())


LLM Catalysis Check:  25%|██▍       | 1752/7144 [53:25<1:55:34,  1.29s/it]

## Store results

In [ ]:
## Create a new column with the results
catalysis_papers_llm = catalysis_papers.add_column("is_catalytic", results)
# select the columns where is_catalytic == True
catalysis_papers_llm_filtered = catalysis_papers_llm.filter(lambda x: x['is_catalytic'] == True)
catalysis_papers_llm_filtered.push_to_hub("DATASET_NAME")

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  2.61ba/s]
Uploading files as bytes or binary IO objects is not supported by Xet Storage. Falling back to HTTP upload.
Uploading the dataset shards: 100%|██████████| 1/1 [00:07<00:00,  7.96s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/amayuelas/LeMat-Synth-Papers-Catalysis/commit/558e8f9e090b66d4dd640e33843aa94127681600', commit_message='Upload dataset', commit_description='', oid='558e8f9e090b66d4dd640e33843aa94127681600', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/amayuelas/LeMat-Synth-Papers-Catalysis', endpoint='https://huggingface.co', repo_type='dataset', repo_id='amayuelas/LeMat-Synth-Papers-Catalysis'), pr_revision=None, pr_num=None)

In [ ]:
## Download and Save some papers in folder
# Download a sample of PDFs from the filtered dataset
download_folder = "../../data/catalysis_pdfs"
os.makedirs(download_folder, exist_ok=True)

sample_size = min(100, len(catalysis_papers_llm_filtered))  # Download 100 or fewer if less available

for idx in range(sample_size):
    pdf_url = catalysis_papers_llm_filtered[idx]['pdf_url']
    filename = os.path.basename(pdf_url.split("?")[0])
    filepath = os.path.join(download_folder, filename)
    try:
        response = requests.get(pdf_url, timeout=30)
        response.raise_for_status()
        with open(filepath, "wb") as f:
            f.write(response.content)
        print(f"Downloaded: {filename}")
    except Exception as e:
        print(f"Failed to download {pdf_url}: {e}")


Downloaded: evolution-of-multicomponent-pd2abcd-cages.pdf
Downloaded: the-origin-of-delayed-polymorphism-in-molecular-crystals-under-mechanochemical-conditions.pdf
Downloaded: epitaxial-thin-film-of-high-entropy-oxide-as-electrocatalyst-for-oxygen-evolution-reaction.pdf
Downloaded: catalysis-as-a-robust-feature-and-catalytic-promiscuity-as-a-recurrent-trait-in-peptide-based-self-replicators.pdf
Downloaded: active-coacervate-droplets-as-a-model-for-membraneless-organelles-and-a-platform-towards-synthetic-life.pdf


In [ ]:
## Load 